# Tutorial 01 — Language models, tokens and baselines

[Lecture notes](../lectures/01_tokenization.md) · [Slides](../slides/01_tokenization.pptx) · [Exercise sheet](../exercises/01_tokenization.md)

**40 minutes, CPU, NumPy only.** You will count the same text in four units, run byte-pair encoding by hand and in code, fit a character bigram baseline on training documents only, and see how a leak and a change of unit each move a reported number. The texts are a few classroom sentences written for this course. Nothing here measures a real tokeniser or a real model.

| Minutes | Step |
|---|---|
| 0–5 | Count one string in words, code points and bytes |
| 5–12 | Byte-pair encoding: count pairs on four Swahili verbs |
| 12–22 | A baseline ladder: uniform, unigram, bigram |
| 22–30 | Two ways to get a better number without a better model |
| 30–40 | Your turn: two functions with tests |

In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)

Python 3.13.1 NumPy 2.1.3 seed 42


## 1. One string, four counts

The first three sentences say "the class starts today" in English, Hausa and Swahili. Then come a Hausa word with a hooked letter (ƙasa, "country"), a Yoruba greeting with tone marks, an Amharic greeting in Ge'ez script, and "café" typed two ways.

Before you run the cell, predict which strings need more bytes than code points, and what NFC does to the two spellings of "café".

In [2]:
import unicodedata
examples = ["The class starts today.", "Darasi ya fara yau.", "Darasa linaanza leo.", "ƙasa", "Ẹ káàárọ̀", "ሰላም", "café", "café"]
for text in examples:
    print(repr(text), {"words": len(text.split()), "codepoints": len(text),
          "bytes": len(text.encode("utf-8")), "NFC_codepoints": len(unicodedata.normalize("NFC", text))})
assert len(examples[-2]) != len(examples[-1])
assert unicodedata.normalize("NFC", examples[-2]) == examples[-1]

'The class starts today.' {'words': 4, 'codepoints': 23, 'bytes': 23, 'NFC_codepoints': 23}
'Darasi ya fara yau.' {'words': 4, 'codepoints': 19, 'bytes': 19, 'NFC_codepoints': 19}
'Darasa linaanza leo.' {'words': 3, 'codepoints': 20, 'bytes': 20, 'NFC_codepoints': 20}
'ƙasa' {'words': 1, 'codepoints': 4, 'bytes': 5, 'NFC_codepoints': 4}
'Ẹ káàárọ̀' {'words': 2, 'codepoints': 9, 'bytes': 17, 'NFC_codepoints': 9}
'ሰላም' {'words': 1, 'codepoints': 3, 'bytes': 9, 'NFC_codepoints': 3}
'café' {'words': 1, 'codepoints': 5, 'bytes': 6, 'NFC_codepoints': 4}
'café' {'words': 1, 'codepoints': 4, 'bytes': 5, 'NFC_codepoints': 4}


The Amharic word is 3 code points and 9 bytes, so a byte-level model spends three positions on each letter. The Yoruba greeting keeps 9 code points after NFC, because ọ̀ has no single code point: NFC joins what it can and leaves the rest as base letter plus mark. Which of these eight strings would a tokeniser trained only on English news have seen often?

## 2. Byte-pair encoding starts by counting pairs

BPE begins with one symbol per character. It counts every adjacent pair of symbols in the training words, merges the most frequent pair into a new symbol, and repeats. The corpus is four Swahili verbs: *ninasoma* (I am reading), *anasema* (she is saying), *unalala* (you are sleeping), *tunakula* (we are eating).

Count the pairs by hand first. Which pair wins, and what does it mean in Swahili?

In [3]:
from collections import Counter
bpe_words = {tuple(w): 1 for w in ["ninasoma", "anasema", "unalala", "tunakula"]}   # symbols -> word count

def pair_counts(words):
    counts = Counter()
    for symbols, n in words.items():
        for a, b in zip(symbols, symbols[1:]):
            counts[(a, b)] += n
    return counts

def apply_pair(symbols, pair):
    """Merge every occurrence of `pair` in a tuple of symbols."""
    out, i = [], 0
    while i < len(symbols):
        if symbols[i:i+2] == pair: out.append(symbols[i] + symbols[i+1]); i += 2
        else: out.append(symbols[i]); i += 1
    return tuple(out)

first = pair_counts(bpe_words)
print(sorted(first.items(), key=lambda kv: (-kv[1], kv[0]))[:6])
print(apply_pair(tuple("ninasoma"), ("n", "a")))
assert first[("n", "a")] == 4 and first[("l", "a")] == 3

[(('n', 'a'), 4), (('l', 'a'), 3), (('a', 'l'), 2), (('a', 's'), 2), (('m', 'a'), 2), (('u', 'n'), 2)]
('n', 'i', 'na', 's', 'o', 'm', 'a')


The pair (n, a) appears 4 times, once in every verb. It is the present-tense marker *-na-*. BPE found it by counting and knows nothing about tense: the runner-up (l, a) is half of *lala* and the end of *kula*. After the merge, *ninasoma* is 7 symbols in place of 8. You will write the full merge step in section 6.

## 3. A baseline ladder, fitted on training documents only

The model is a character bigram with add-α smoothing. The vocabulary comes from the three training documents, plus an unknown symbol and document boundaries. The development document plays no part in the vocabulary or the counts.

Predict the order of the three perplexities before running: uniform, unigram, bigram.

In [4]:
train_docs = ["the class starts today", "the lab starts today", "the class ends soon"]
dev_docs = ["the lab ends soon"]
vocab = ["<unk>", "<bos>", "<eos>"] + sorted(set("".join(train_docs)))
ids = {c:i for i,c in enumerate(vocab)}
def encode(doc):
    return [ids["<bos>"]] + [ids.get(c, 0) for c in doc] + [ids["<eos>"]]
counts = np.zeros((len(vocab), len(vocab)))
for doc in train_docs:
    seq = encode(doc)
    for a,b in zip(seq,seq[1:]): counts[a,b] += 1
alpha = 0.5
probs = (counts + alpha) / (counts.sum(axis=1,keepdims=True) + alpha*len(vocab))
assert np.allclose(probs.sum(axis=1),1)
def nll(doc):
    seq=encode(doc)
    return -sum(np.log(probs[a,b]) for a,b in zip(seq,seq[1:])),len(seq)-1
loss,n = nll(dev_docs[0])
print({"dev_nll":loss,"predicted_tokens":n,"dev_perplexity":np.exp(loss/n),"uniform_perplexity":len(vocab)})
unigram = (counts.sum(0) + alpha) / (counts.sum() + alpha*len(vocab))       # ignores the previous character
uni_loss = -sum(np.log(unigram[b]) for b in encode(dev_docs[0])[1:])
print("dev perplexity: uniform", len(vocab), " unigram", round(float(np.exp(uni_loss/n)), 3), " bigram", round(float(np.exp(loss/n)), 3))
assert np.exp(loss/n) < np.exp(uni_loss/n) < len(vocab)

{'dev_nll': np.float64(33.17008795105448), 'predicted_tokens': 18, 'dev_perplexity': np.float64(6.314083811403138), 'uniform_perplexity': 17}
dev perplexity: uniform 17  unigram 14.485  bigram 6.314


Each rung uses more of the context and the perplexity falls. A new model has to beat the bigram, the strongest cheap rung, before its number means anything. The development document is 17 characters long, so these numbers illustrate the method and nothing more.

## 4. Choose the smoothing strength on development data

Sweep α on two development documents. Every character transition in the first one also occurs in the training documents. The second one, "the test ends today", has two transitions the training documents never show: t→e and e→s. Predict the best α for each before running.

In [5]:
dev_unseen = "the test ends today"
for alpha in [0.01,0.1,0.5,1.0]:
    probs=(counts+alpha)/(counts.sum(axis=1,keepdims=True)+alpha*len(vocab))
    loss,n=nll(dev_docs[0])
    loss2,n2=nll(dev_unseen)
    print("alpha",alpha,"dev PPL",round(float(np.exp(loss/n)),3)," with two unseen transitions",round(float(np.exp(loss2/n2)),3))
assert np.isclose(np.exp(-np.mean(np.log([.5,.25,.25]))),3.174802103936399)

alpha 0.01 dev PPL 2.901  with two unseen transitions 6.101
alpha 0.1 dev PPL 3.764  with two unseen transitions 5.441
alpha 0.5 dev PPL 6.314  with two unseen transitions 7.242
alpha 1.0 dev PPL 8.261  with two unseen transitions 8.903


When every transition was seen in training, the smallest α wins: 2.901 at α = 0.01. With two unseen transitions the best α is 0.1 (5.441), and α = 0.01 is worse (6.101) because each unseen transition gets almost no probability. The right α depends on how much of the held-out text is new, which is why you choose it on development data. Once you have chosen α there, that perplexity is a selection score. A test document, opened once, gives the number to report.

## 5. Two ways to get a better number without a better model

**A leak.** Fit the counts on training plus development documents, then score the development document.

**A change of unit.** Keep the model and its total NLL fixed, and divide by words in place of characters.

Predict the direction of each change.

In [6]:
def fit_and_score(fit_docs, alpha=0.5):
    c = np.zeros((len(vocab), len(vocab)))
    for doc in fit_docs:
        seq = encode(doc)
        for a, b in zip(seq, seq[1:]): c[a, b] += 1
    p = (c + alpha) / (c.sum(axis=1, keepdims=True) + alpha*len(vocab))
    seq = encode(dev_docs[0])
    return -sum(np.log(p[a, b]) for a, b in zip(seq, seq[1:]))

clean, leaky = fit_and_score(train_docs), fit_and_score(train_docs + dev_docs)
print("dev PPL, fitted on train only:", round(float(np.exp(clean/n)), 3))
print("dev PPL, fitted on train + dev:", round(float(np.exp(leaky/n)), 3))
n_words = len(dev_docs[0].split())
print("same model, total NLL", round(float(clean), 3), "nats")
print("  per character:", round(float(np.exp(clean/n)), 3), " per word:", round(float(np.exp(clean/n_words)), 1))
assert leaky < clean

dev PPL, fitted on train only: 6.314
dev PPL, fitted on train + dev: 4.751
same model, total NLL 33.17 nats
  per character: 6.314  per word: 3993.9


The leak lowers perplexity from 6.314 to 4.751 with no change to the method. Dividing the same 33.17 nats by 4 words in place of 18 character predictions turns 6.3 into 3,994. Neither change tells you anything new about the model. Compare models with different tokenisers in a unit that the tokeniser cannot change: bits per byte of the original text.

## 6. Your turn

Replace each `raise` with your code, then run the test cell under it. Do not edit the tests.

**A. One BPE merge step.** `words` maps a tuple of symbols to a word count, as `bpe_words` does. Find the most frequent adjacent pair. Break ties by taking the pair that sorts first, so `("m", "a")` beats `("u", "na")`. Return the pair and a new dictionary in which every occurrence of the pair is one merged symbol.

In [ ]:
# YOUR TURN
def bpe_merge_step(words):
    """Return (best_pair, new_words) after merging the most frequent adjacent pair."""
    raise NotImplementedError("your turn")

In [8]:
pair, merged = bpe_merge_step(bpe_words)
assert pair == ("n", "a") and ("n", "i", "na", "s", "o", "m", "a") in merged
pair2, merged2 = bpe_merge_step(merged)
assert pair2 == ("l", "a") and ("u", "na", "la", "la") in merged2
assert bpe_merge_step(merged2)[0] == ("m", "a")            # three pairs tie at 2; the rule picks this one
assert sum(merged2.values()) == 4 and len(bpe_words) == 4   # the input is not changed

def learn_bpe(docs, n_merges):
    words, merges = Counter(tuple(w) for d in docs for w in d.split()), []
    for _ in range(n_merges):
        if all(len(s) == 1 for s in words): break
        pair, words = bpe_merge_step(words)
        merges.append(pair)
    return merges

def tokenize(text, merges):
    out = []
    for word in text.split():
        symbols = tuple(word)
        for pair in merges: symbols = apply_pair(symbols, pair)
        out.extend(symbols)
    return out

merges = learn_bpe(train_docs, 20)
print("first merges learned on the English training documents:", merges[:6])
for name, text in [("English, held out", dev_docs[0]), ("Hausa", "darasi ya fara yau"), ("Swahili", "darasa linaanza leo")]:
    toks = tokenize(text, merges)
    print(f"{name:18s} {len(toks):2d} tokens for {len(text.replace(' ', '')):2d} letters:", " ".join(toks))
print("A passed")

first merges learned on the English training documents: [('h', 'e'), ('l', 'a'), ('t', 'he'), ('a', 'r'), ('a', 'y'), ('ar', 't')]
English, held out   6 tokens for 14 letters: the lab ends s o on
Hausa              13 tokens for 15 letters: d ar a s i y a f ar a y a u
Swahili            16 tokens for 17 letters: d ar a s a l i n a a n z a l e o
A passed


The tokeniser was trained on three English sentences. It covers the held-out English sentence in a few tokens and spells the Hausa and Swahili sentences out almost letter by letter. Production tokenisers show the same effect at scale for languages with little text in their training data (Petrov et al., 2023). This toy shows the mechanism and not its size.

**B. Bits per byte.** Given the natural-log probability of each token and the original text, return the total number of bits divided by the number of UTF-8 bytes.

In [ ]:
# YOUR TURN
def bits_per_byte(token_logprobs, text):
    """Total negative log2-likelihood of the tokens divided by the UTF-8 byte length of `text`."""
    raise NotImplementedError("your turn")

In [10]:
assert np.isclose(bits_per_byte(np.log([0.5, 0.25, 0.25]), "twelve bytes"), 5 / 12)
assert np.isclose(bits_per_byte(np.log([0.5, 0.25, 0.25]), "ሰላም"), 5 / 9)          # 3 letters, 9 bytes
seq = encode(dev_docs[0])
probs = (counts + 0.1) / (counts.sum(axis=1, keepdims=True) + 0.1*len(vocab))
logp = [np.log(probs[a, b]) for a, b in zip(seq, seq[1:])]
print("bigram, alpha 0.1, on the development document:", round(float(bits_per_byte(logp, dev_docs[0])), 3), "bits per byte")
print("uniform over the vocabulary:", round(float(bits_per_byte([np.log(1/len(vocab))]*len(logp), dev_docs[0])), 3), "bits per byte")
print("B passed")

bigram, alpha 0.1, on the development document: 2.025 bits per byte
uniform over the vocabulary: 4.328 bits per byte
B passed


## Discussion

1. A tokeniser uses twice as many tokens for Hausa as for English on your sample. What can you conclude, and what can you not?
2. Why is fitting the vocabulary before splitting the data a leak?
3. Two sentences come from the same news article. Which splits may they go in?

## From the toy to your task

Run the same three checks on your SemEval task's training file tonight: count the rows per label and per language, compute the majority-class score in the task's metric, and look for the same text or the same image in more than one split. The exercise sheet asks for these numbers.

Submit this notebook with both tests passing and a 150–250-word note: the two numbers from section 5, which of them a reader could be fooled by, and how your team will avoid both in its paper.